# LeetCode 1380 — Lucky Numbers in a Matrix bằng Python

**Link đề:** https://leetcode.com/problems/lucky-numbers-in-a-matrix/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix

```python
class Solution:
    def luckyNumbers(self, matrix: list[list[int]]) -> list[int]:
        ...
```

### Mục tiêu học
1. Duyệt ma trận theo **hàng** và theo **cột**; dùng `min`, `max`, `zip(*matrix)` (chuyển vị).
2. Từ lời giải vét cạn `O(m·n·(m+n))`, rút ra kỹ thuật **tiền xử lý** (precompute) min hàng / max cột để còn `O(m·n)`.
3. Dùng **set** và phép **giao** (`&`) để viết lời giải gọn.
4. Chứng minh một tính chất bất ngờ: với các phần tử **phân biệt**, có **nhiều nhất một** lucky number → lời giải "max của các min hàng == min của các max cột".
5. Trace, chứng minh, phân tích Time/Space, test đối chiếu và tự giải biến thể (điểm yên ngựa).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho ma trận `m x n` gồm các số **phân biệt**. Trả về mọi **lucky number** (số may mắn), thứ tự bất kỳ.

Một phần tử là lucky number nếu nó vừa là **nhỏ nhất trong hàng** của nó, vừa là **lớn nhất trong cột** của nó.

**Ví dụ 1**
```text
 3   7   8
 9  11  13
15  16  17
```
Output `[15]`: 15 nhỏ nhất hàng 2 (`15 < 16 < 17`) và lớn nhất cột 0 (`15 > 9 > 3`).

**Ví dụ 2:** `[[1,10,4,2],[9,3,8,7],[15,16,17,12]]` → `[12]`.
**Ví dụ 3:** `[[7,8],[1,2]]` → `[7]`.

**Constraints:** `1 <= m, n <= 50`; `1 <= matrix[i][j] <= 10^5`; mọi phần tử **phân biệt**.

In [1]:
mat1 = [[3, 7, 8], [9, 11, 13], [15, 16, 17]]
mat2 = [[1, 10, 4, 2], [9, 3, 8, 7], [15, 16, 17, 12]]
mat3 = [[7, 8], [1, 2]]
for row in mat2:
    print(row)
print("m =", len(mat2), "n =", len(mat2[0]))

[1, 10, 4, 2]
[9, 3, 8, 7]
[15, 16, 17, 12]
m = 3 n = 4


## 2. Nền tảng: hàng, cột, `min`, `max`

- Hàng `i`: `matrix[i]` — một list, `min(matrix[i])` là nhỏ nhất của hàng.
- Cột `j`: không có sẵn list; ta tự lấy `[matrix[i][j] for i in range(m)]`.

In [2]:
i, j = 2, 3
row = mat2[i]
col = [mat2[r][j] for r in range(len(mat2))]
print("Hàng", i, ":", row, "→ min =", min(row))
print("Cột", j, ":", col, "→ max =", max(col))
print("mat2[2][3] =", mat2[2][3], "→ vừa là min hàng vừa là max cột?", mat2[2][3] == min(row) == max(col))

Hàng 2 : [15, 16, 17, 12] → min = 12
Cột 3 : [2, 7, 12] → max = 12
mat2[2][3] = 12 → vừa là min hàng vừa là max cột? True


`a == b == c` là **so sánh chuỗi** của Python: tương đương `a == b and b == c`.

### 2.1. `zip(*matrix)` — lấy tất cả các cột cùng lúc (chuyển vị)

- `zip(a, b, c)` ghép phần tử cùng vị trí: `zip([1,2],[3,4])` → `(1,3), (2,4)`.
- `*matrix` **rải** các hàng thành từng đối số: `zip(*matrix)` = `zip(hàng0, hàng1, ...)` → các tuple chính là **các cột**.

In [3]:
print(list(zip([1, 2], [3, 4])))
cols = list(zip(*mat1))
print("Các cột của mat1:", cols)
print("max mỗi cột:", [max(c) for c in cols])
print("min mỗi hàng:", [min(r) for r in mat1])

[(1, 3), (2, 4)]
Các cột của mat1: [(3, 9, 15), (7, 11, 16), (8, 13, 17)]
max mỗi cột: [15, 16, 17]
min mỗi hàng: [3, 9, 15]


## 3. Cách 1 — Vét cạn: kiểm tra từng ô

Với mỗi ô `(i, j)`: tính min hàng `i` và max cột `j`, nếu `matrix[i][j]` bằng cả hai thì là lucky.

In [4]:
def lucky_brute(matrix):
    m, n = len(matrix), len(matrix[0])
    res = []
    for i in range(m):
        for j in range(n):
            v = matrix[i][j]
            row_min = min(matrix[i])
            col_max = max(matrix[r][j] for r in range(m))
            if v == row_min and v == col_max:
                res.append(v)
    return res

print(lucky_brute(mat1), lucky_brute(mat2), lucky_brute(mat3))

[15] [12] [7]


### Độ phức tạp và nút thắt

- Mỗi ô tốn `O(n)` để tính min hàng và `O(m)` để tính max cột → tổng `O(m·n·(m + n))`. Với `m = n = 50`: `2500 × 100 = 250.000` phép — chạy được.
- **Nút thắt:** min của hàng `i` được tính lại `n` lần (mỗi ô trong hàng một lần), max của cột `j` tính lại `m` lần. Kết quả **không đổi** — ta đang lặp lại công việc.

**Ý tưởng cải tiến:** tính **một lần** min của mỗi hàng và max của mỗi cột, lưu vào hai list (tiền xử lý), rồi tra cứu `O(1)`.

In [5]:
calls = {"row_min": 0, "col_max": 0}

def lucky_brute_counted(matrix):
    m, n = len(matrix), len(matrix[0])
    for i in range(m):
        for j in range(n):
            calls["row_min"] += 1
            calls["col_max"] += 1
    return calls

print("Ma trận 50x50 — số lần tính lại min hàng / max cột:", lucky_brute_counted([[0] * 50 for _ in range(50)]))
print("Trong khi chỉ có 50 hàng và 50 cột khác nhau.")

Ma trận 50x50 — số lần tính lại min hàng / max cột: {'row_min': 2500, 'col_max': 2500}
Trong khi chỉ có 50 hàng và 50 cột khác nhau.


## 4. Cách 2 — Tiền xử lý min hàng và max cột

1. `row_min[i] = min(matrix[i])` cho mọi `i` — `O(m·n)`.
2. `col_max[j] = max(cột j)` cho mọi `j` — `O(m·n)`.
3. Duyệt mọi ô, `matrix[i][j]` là lucky ⇔ `matrix[i][j] == row_min[i] == col_max[j]`.

In [6]:
def lucky_precompute(matrix):
    m, n = len(matrix), len(matrix[0])
    row_min = [min(row) for row in matrix]
    col_max = [max(matrix[i][j] for i in range(m)) for j in range(n)]
    res = []
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == row_min[i] == col_max[j]:
                res.append(matrix[i][j])
    return res

print(lucky_precompute(mat1), lucky_precompute(mat2), lucky_precompute(mat3))

[15] [12] [7]


## 5. TRACE — Cách 2 trên ví dụ 2

```text
        j=0  j=1  j=2  j=3   row_min
i=0      1   10    4    2      1
i=1      9    3    8    7      3
i=2     15   16   17   12     12
col_max 15   16   17   12
```

Ô nào có giá trị bằng **cả** `row_min` của hàng **và** `col_max` của cột? Chỉ `(2, 3) = 12`.

In [7]:
def trace_lucky(matrix):
    m, n = len(matrix), len(matrix[0])
    row_min = [min(row) for row in matrix]
    col_max = [max(c) for c in zip(*matrix)]
    print("row_min =", row_min)
    print("col_max =", col_max)
    res = []
    for i in range(m):
        for j in range(n):
            v = matrix[i][j]
            is_min = v == row_min[i]
            is_max = v == col_max[j]
            if is_min or is_max:          # chỉ in các ô "đáng chú ý"
                tag = "LUCKY" if is_min and is_max else ("min hàng" if is_min else "max cột")
                print(f"  ({i},{j}) = {v:>2}: min hàng? {is_min!s:<5} max cột? {is_max!s:<5} → {tag}")
            if is_min and is_max:
                res.append(v)
    print("Kết quả:", res)
    return res

trace_lucky(mat2)

row_min = [1, 3, 12]
col_max = [15, 16, 17, 12]
  (0,0) =  1: min hàng? True  max cột? False → min hàng
  (1,1) =  3: min hàng? True  max cột? False → min hàng
  (2,0) = 15: min hàng? False max cột? True  → max cột
  (2,1) = 16: min hàng? False max cột? True  → max cột
  (2,2) = 17: min hàng? False max cột? True  → max cột
  (2,3) = 12: min hàng? True  max cột? True  → LUCKY
Kết quả: [12]


[12]

Trace in những ô **là min hàng hoặc max cột** để thấy: các min hàng là `1, 3, 12`, các max cột là `15, 16, 17, 12` — chỉ `12` thuộc cả hai.

`{x!s:<5}` = đổi sang chuỗi (`!s`) rồi căn trái rộng 5 — để `True`/`False` thẳng cột.

## 6. Cách 3 — Giao hai tập hợp

Vì các phần tử **phân biệt**, một giá trị xác định đúng một ô. Vậy:

```text
lucky = {các min hàng} ∩ {các max cột}
```

Nếu giá trị `v` vừa là min của một hàng nào đó, vừa là max của một cột nào đó → do phân biệt, hai vai trò đó nói về **cùng một ô** → ô đó lucky.

`set` là tập hợp không trùng lặp; `A & B` là **giao** (phần tử chung).

In [8]:
A = {1, 3, 12}
B = {15, 16, 17, 12}
print(A & B, A | B, A - B)       # giao, hợp, hiệu

def lucky_sets(matrix):
    row_mins = {min(row) for row in matrix}
    col_maxs = {max(col) for col in zip(*matrix)}
    return list(row_mins & col_maxs)

print(lucky_sets(mat1), lucky_sets(mat2), lucky_sets(mat3))

{12} {16, 1, 17, 3, 12, 15} {1, 3}
[15] [12] [7]


`{min(row) for row in matrix}` là **set comprehension** (giống list comprehension nhưng dùng `{}` và tự bỏ trùng).

## 7. Insight — có **nhiều nhất một** lucky number

**Mệnh đề:** nếu mọi phần tử phân biệt, ma trận có **tối đa một** lucky number.

**Chứng minh:** giả sử có hai lucky number khác nhau: `a` ở ô `(r1, c1)` và `b` ở ô `(r2, c2)`.
- `a` là min hàng `r1` nên `a <= matrix[r1][c2]`. `b` là max cột `c2` nên `matrix[r1][c2] <= b`. Vậy `a <= b`.
- Đối xứng: `b` là min hàng `r2` nên `b <= matrix[r2][c1]`; `a` là max cột `c1` nên `matrix[r2][c1] <= a`. Vậy `b <= a`.
- Suy ra `a == b`, mâu thuẫn với "khác nhau" (phân biệt). (Nếu `r1 == r2` hoặc `c1 == c2`, lập luận vẫn đúng vì một hàng/cột chỉ có một min/max.)

**Hệ quả mạnh hơn:** gọi `X = max(row_min)` và `Y = min(col_max)`. Luôn có `X <= Y`, và **có lucky ⇔ `X == Y`**; khi đó lucky chính là `X`.

In [9]:
for mat in (mat1, mat2, mat3, [[3, 6], [7, 1], [5, 2], [4, 8]]):
    row_min = [min(r) for r in mat]
    col_max = [max(c) for c in zip(*mat)]
    X, Y = max(row_min), min(col_max)
    print(f"row_min={row_min}, col_max={col_max} → X={X}, Y={Y}, X==Y? {X == Y} | lucky = {lucky_precompute(mat)}")

row_min=[3, 9, 15], col_max=[15, 16, 17] → X=15, Y=15, X==Y? True | lucky = [15]
row_min=[1, 3, 12], col_max=[15, 16, 17, 12] → X=12, Y=12, X==Y? True | lucky = [12]
row_min=[7, 1], col_max=[7, 8] → X=7, Y=7, X==Y? True | lucky = [7]
row_min=[3, 1, 2, 4], col_max=[7, 8] → X=4, Y=7, X==Y? False | lucky = []


### 7.1. Vì sao `X <= Y` luôn đúng?

Lấy hàng `i*` đạt `X = row_min[i*]` và cột `j*` đạt `Y = col_max[j*]`. Xét ô giao `v = matrix[i*][j*]`:
- `X = min hàng i*` ≤ `v`;
- `v` ≤ `max cột j*` = `Y`.

Vậy `X <= v <= Y`. Nếu `X == Y` thì `v == X == Y`, nghĩa là `v` vừa là min hàng `i*` vừa là max cột `j*` → lucky. Ngược lại nếu có lucky `L` ở `(r, c)` thì `L = row_min[r] <= X` và `L = col_max[c] >= Y`, kết hợp `X <= Y` cho `X == Y == L`.

## 8. Cách 4 — `max(row_min) == min(col_max)`

In [10]:
def lucky_maxmin(matrix):
    X = max(min(row) for row in matrix)
    Y = min(max(col) for col in zip(*matrix))
    return [X] if X == Y else []

print(lucky_maxmin(mat1), lucky_maxmin(mat2), lucky_maxmin(mat3), lucky_maxmin([[3, 6], [7, 1], [5, 2], [4, 8]]))

[15] [12] [7] []


Cách này không cần lưu cả hai list, không cần duyệt lần hai — chỉ hai số `X`, `Y`. Nhưng nó **dựa vào giả thiết phân biệt**; nếu đề bỏ giả thiết đó (xem biến thể), hãy quay lại Cách 2.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def luckyNumbers(self, matrix: list[list[int]]) -> list[int]:
        max_of_row_mins = max(min(row) for row in matrix)
        min_of_col_maxs = min(max(col) for col in zip(*matrix))
        return [max_of_row_mins] if max_of_row_mins == min_of_col_maxs else []

sol = Solution()
print(sol.luckyNumbers([[3,7,8],[9,11,13],[15,16,17]]))          # [15]
print(sol.luckyNumbers([[1,10,4,2],[9,3,8,7],[15,16,17,12]]))    # [12]
print(sol.luckyNumbers([[7,8],[1,2]]))                           # [7]

[15]
[12]
[7]


Bản tiền xử lý (Cách 2) — tổng quát hơn, cũng Accepted:

In [12]:
class SolutionPrecompute:
    def luckyNumbers(self, matrix: list[list[int]]) -> list[int]:
        row_min = [min(row) for row in matrix]
        col_max = [max(col) for col in zip(*matrix)]
        return [v for i, row in enumerate(matrix) for j, v in enumerate(row)
                if v == row_min[i] == col_max[j]]

print(SolutionPrecompute().luckyNumbers([[1,10,4,2],[9,3,8,7],[15,16,17,12]]))

[12]


List comprehension **hai vòng `for`**: `[... for i, row in ... for j, v in ...]` — vòng đầu là vòng ngoài, vòng sau là vòng trong, giống viết hai `for` lồng nhau.

## 10. Vì sao đúng? Độ phức tạp

- **Cách 2** đúng trực tiếp theo định nghĩa: `row_min[i]`, `col_max[j]` là đúng min/max, và ô được chọn ⇔ thoả cả hai.
- **Cách 3** đúng nhờ tính phân biệt (một giá trị ↔ một ô).
- **Cách 4** đúng nhờ mệnh đề mục 7 và 7.1.

Gọi `m` = số hàng, `n` = số cột.

| Cách | Thời gian | Bộ nhớ phụ | Cần "phân biệt"? |
|---|---|---|---|
| 1. Vét cạn | `O(m·n·(m+n))` | `O(1)` | không |
| 2. Tiền xử lý | `O(m·n)` | `O(m + n)` | không |
| 3. Giao set | `O(m·n)` | `O(m + n)` | **có** |
| 4. max-min | `O(m·n)` | `O(1)`* | **có** |

\* `zip(*matrix)` tạo các tuple cột (tạm thời `O(m·n)` nếu gom hết); có thể duyệt cột bằng chỉ số để giữ `O(1)` phụ (xem bài tập).

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [13]:
import random

def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.luckyNumbers([[3,7,8],[9,11,13],[15,16,17]]), [15])
check("đề 2", sol.luckyNumbers([[1,10,4,2],[9,3,8,7],[15,16,17,12]]), [12])
check("đề 3", sol.luckyNumbers([[7,8],[1,2]]), [7])
check("1x1", sol.luckyNumbers([[5]]), [5])
check("một hàng", sol.luckyNumbers([[4, 2, 9]]), [2])
check("một cột", sol.luckyNumbers([[4], [2], [9]]), [9])
check("không có lucky", sol.luckyNumbers([[3, 6], [7, 1], [5, 2], [4, 8]]), [])

PASS đề 1: [15]
PASS đề 2: [12]
PASS đề 3: [7]
PASS 1x1: [5]
PASS một hàng: [2]
PASS một cột: [9]
PASS không có lucky: []


`check` so sánh sau khi `sorted` vì đề cho phép **thứ tự bất kỳ**.

Ma trận một hàng: lucky = min của hàng (mỗi cột chỉ có 1 phần tử nên nó tự là max cột). Một cột: lucky = max của cột.

In [14]:
def random_distinct_matrix(rng, m, n):
    values = rng.sample(range(1, 10**5 + 1), m * n)     # m*n số phân biệt
    return [values[i * n:(i + 1) * n] for i in range(m)]

methods = [lucky_brute, lucky_precompute, lucky_sets, lucky_maxmin,
           Solution().luckyNumbers, SolutionPrecompute().luckyNumbers]
rng = random.Random(1380)
found = 0
for _ in range(500):
    mat = random_distinct_matrix(rng, rng.randint(1, 6), rng.randint(1, 6))
    expected = sorted(lucky_brute(mat))
    assert len(expected) <= 1                            # mệnh đề mục 7
    found += len(expected)
    for f in methods:
        assert sorted(f(mat)) == expected, (f, mat)
print(f"500 ma trận ngẫu nhiên × {len(methods)} cách: PASS ({found} ma trận có lucky number)")

500 ma trận ngẫu nhiên × 6 cách: PASS (223 ma trận có lucky number)


`rng.sample(population, k)` lấy `k` phần tử **không lặp** từ `population` — đảm bảo ma trận phân biệt như đề. Dòng `assert len(expected) <= 1` kiểm chứng thực nghiệm mệnh đề "nhiều nhất một".

## 12. Lỗi người mới hay gặp

### Lỗi 1: Nhầm min/max (max hàng, min cột)

In [15]:
def wrong_swapped(matrix):
    row_max = [max(r) for r in matrix]
    col_min = [min(c) for c in zip(*matrix)]
    return [v for i, r in enumerate(matrix) for j, v in enumerate(r) if v == row_max[i] == col_min[j]]

print("Sai :", wrong_swapped(mat1), "| Đúng:", lucky_precompute(mat1))

Sai : [8] | Đúng: [15]


### Lỗi 2: Lấy cột bằng `matrix[j]`
`matrix[j]` là **hàng** `j`, không phải cột. Với ma trận không vuông có thể gây `IndexError`; với ma trận vuông thì ra kết quả sai âm thầm.

In [16]:
def wrong_column(matrix):
    res = []
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            v = matrix[i][j]
            if v == min(matrix[i]) and v == max(matrix[j]):   # SAI: matrix[j] là hàng
                res.append(v)
    return res

print("Sai :", wrong_column(mat3), "| Đúng:", lucky_precompute(mat3))
try:
    wrong_column(mat2)
except IndexError as e:
    print("Ma trận 3x4 → IndexError:", e)

Sai : [] | Đúng: [7]
Ma trận 3x4 → IndexError: list index out of range


### Lỗi 3: Dùng `zip(matrix)` (thiếu dấu `*`)
`zip(matrix)` coi cả ma trận là **một** đối số → mỗi phần tử là tuple chứa **một hàng**, không phải cột.

In [17]:
print("zip(matrix) :", list(zip(mat3)))
print("zip(*matrix):", list(zip(*mat3)))

zip(matrix) : [([7, 8],), ([1, 2],)]
zip(*matrix): [(7, 1), (8, 2)]


### Lỗi 4: Áp dụng Cách 4 khi phần tử **có thể trùng**
Nếu đề không đảm bảo phân biệt, có thể có **nhiều** ô cùng giá trị thoả — Cách 4 chỉ trả một số. Đọc kỹ constraints trước khi dùng mẹo.

In [18]:
dup = [[1, 1], [1, 1]]
print("max-min trả:", lucky_maxmin(dup), "| tiền xử lý trả (mỗi ô một lần):", lucky_precompute(dup))

max-min trả: [1] | tiền xử lý trả (mỗi ô một lần): [1, 1, 1, 1]


### Mẹo nhận dạng pattern

- Điều kiện kiểu **"nhỏ nhất trong hàng VÀ lớn nhất trong cột"** → tiền xử lý `row_min`, `col_max` (hoặc tương tự `row_sum`, `col_count`…).
- Cần các cột của ma trận trong Python → `zip(*matrix)`.
- Giá trị **phân biệt** → nghĩ tới set, và tìm tính chất "duy nhất" để rút gọn.
- Bất đẳng thức **max-min ≤ min-max** là một kết quả kinh điển (xuất hiện trong lý thuyết trò chơi — điểm yên ngựa).
- Bài anh em: *Special Positions in a Binary Matrix* (1582), *Toeplitz Matrix* (766), *Transpose Matrix* (867), *Count Servers that Communicate* (1267).

## 13. Biến thể

### 13.1. Đảo vai: lớn nhất hàng VÀ nhỏ nhất cột
Đối xứng hoàn toàn: `min(max hàng) == max(min cột)`.

In [19]:
def anti_lucky(matrix):
    X = min(max(row) for row in matrix)
    Y = max(min(col) for col in zip(*matrix))
    return [X] if X == Y else []

print(anti_lucky([[1, 2], [3, 4]]), anti_lucky([[3, 7, 8], [9, 11, 13], [15, 16, 17]]))

[2] [8]


### 13.2. Trả về **vị trí** thay vì giá trị

In [20]:
def lucky_positions(matrix):
    row_min = [min(r) for r in matrix]
    col_max = [max(c) for c in zip(*matrix)]
    return [(i, j) for i, r in enumerate(matrix) for j, v in enumerate(r) if v == row_min[i] == col_max[j]]

print(lucky_positions(mat2))

[(2, 3)]


### 13.3. Special Positions (LeetCode 1582)
Đếm ô bằng `1` mà hàng và cột của nó chỉ có đúng một số `1`. Cùng kỹ thuật: tiền xử lý tổng mỗi hàng / mỗi cột.

In [21]:
def num_special(mat):
    row_sum = [sum(r) for r in mat]
    col_sum = [sum(c) for c in zip(*mat)]
    return sum(1 for i, r in enumerate(mat) for j, v in enumerate(r)
               if v == 1 and row_sum[i] == 1 and col_sum[j] == 1)

print(num_special([[1, 0, 0], [0, 0, 1], [1, 0, 0]]))   # 1
print(num_special([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))   # 3

1
3


## 14. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Tính tay
Với `[[10, 20, 30], [5, 25, 35], [40, 45, 50]]`, tính `row_min`, `col_max`, `X`, `Y`. Có lucky number không?

### Bài 2 — Tiền xử lý không dùng `zip`
Viết `exercise_col_max(matrix)` trả list max mỗi cột chỉ dùng chỉ số.

In [22]:
def exercise_col_max(matrix):
    # TODO: hai vòng for, j ngoài, i trong
    pass

# Gợi ý test:
# assert exercise_col_max([[1, 10, 4, 2], [9, 3, 8, 7], [15, 16, 17, 12]]) == [15, 16, 17, 12]

### Bài 3 — Hoàn thiện luckyNumbers (Cách 2)

In [23]:
def exercise_lucky(matrix):
    # TODO 1: row_min
    # TODO 2: col_max
    # TODO 3: duyệt ô, chọn ô bằng cả hai
    pass

# Gợi ý test:
# assert exercise_lucky([[7, 8], [1, 2]]) == [7]

### Bài 4 — Cách 4 với bộ nhớ phụ `O(1)`
Tính `X = max(min hàng)` và `Y = min(max cột)` mà **không** tạo list/tuple nào (không `zip`, không list comprehension).

In [24]:
def exercise_lucky_o1(matrix):
    # TODO
    pass

# Gợi ý test:
# assert exercise_lucky_o1([[1, 10, 4, 2], [9, 3, 8, 7], [15, 16, 17, 12]]) == [12]
# assert exercise_lucky_o1([[3, 6], [7, 1], [5, 2], [4, 8]]) == []

### Bài 5 — Điểm yên ngựa cho phép trùng
Ma trận **có thể có giá trị trùng**. Trả về list **vị trí** `(i, j)` mà `matrix[i][j]` là min hàng **và** max cột.

In [25]:
def exercise_saddle_points(matrix):
    # TODO
    pass

# Gợi ý test:
# assert exercise_saddle_points([[1, 1], [1, 1]]) == [(0, 0), (0, 1), (1, 0), (1, 1)]
# assert exercise_saddle_points([[2, 2], [1, 3]]) == [(0, 0)]

### Bài 6 — Mini challenge: chứng minh bằng thực nghiệm
Viết `exercise_check_inequality(trials)` sinh `trials` ma trận ngẫu nhiên (cho phép trùng) và kiểm tra `max(min hàng) <= min(max cột)` luôn đúng; trả `True` nếu không có phản ví dụ.

In [26]:
def exercise_check_inequality(trials):
    # TODO
    pass

# Gợi ý test:
# assert exercise_check_inequality(300) is True

## 15. Gợi ý 3 mức

**Bài 1** — (1) Min mỗi hàng: 10, 5, 40. (2) Max mỗi cột: 40, 45, 50. (3) `X = 40`, `Y = 40`.

**Bài 2** — (1) Mỗi cột một giá trị. (2) Khởi tạo `best = matrix[0][j]`. (3) `for i in range(1, m): best = max(best, matrix[i][j])`.

**Bài 3** — (1) Xem mục 4. (2) `row_min = [min(r) for r in matrix]`. (3) Điều kiện `v == row_min[i] == col_max[j]`.

**Bài 4** — (1) Hai biến chạy. (2) `X` bắt đầu `-inf`, cập nhật bằng min từng hàng (tính bằng vòng lặp). (3) `Y` bắt đầu `+inf`, với mỗi cột tính max bằng vòng lặp qua `i`.

**Bài 5** — (1) Không dùng mẹo duy nhất. (2) Dùng Cách 2 nhưng trả vị trí. (3) Mục 13.2.

**Bài 6** — (1) Sinh ma trận với giá trị nhỏ để có trùng. (2) Tính `X`, `Y` như Cách 4. (3) Nếu `X > Y` thì trả `False`.

## 16. Đáp án tham khảo

**Bài 1:** `row_min = [10, 5, 40]`, `col_max = [40, 45, 50]`, `X = 40 = Y` → lucky `[40]` (ô `(2, 0)`).

In [27]:
assert Solution().luckyNumbers([[10, 20, 30], [5, 25, 35], [40, 45, 50]]) == [40]

def answer_col_max(matrix):
    m, n = len(matrix), len(matrix[0])
    res = []
    for j in range(n):
        best = matrix[0][j]
        for i in range(1, m):
            if matrix[i][j] > best:
                best = matrix[i][j]
        res.append(best)
    return res

def answer_lucky(matrix):
    row_min = [min(r) for r in matrix]
    col_max = answer_col_max(matrix)
    res = []
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            if matrix[i][j] == row_min[i] == col_max[j]:
                res.append(matrix[i][j])
    return res

def answer_lucky_o1(matrix):
    m, n = len(matrix), len(matrix[0])
    X = float("-inf")
    for i in range(m):
        rmin = matrix[i][0]
        for j in range(1, n):
            if matrix[i][j] < rmin:
                rmin = matrix[i][j]
        if rmin > X:
            X = rmin
    Y = float("inf")
    for j in range(n):
        cmax = matrix[0][j]
        for i in range(1, m):
            if matrix[i][j] > cmax:
                cmax = matrix[i][j]
        if cmax < Y:
            Y = cmax
    return [X] if X == Y else []

def answer_saddle_points(matrix):
    row_min = [min(r) for r in matrix]
    col_max = [max(c) for c in zip(*matrix)]
    return [(i, j) for i, r in enumerate(matrix) for j, v in enumerate(r) if v == row_min[i] == col_max[j]]

def answer_check_inequality(trials):
    rng = random.Random(2024)
    for _ in range(trials):
        m, n = rng.randint(1, 5), rng.randint(1, 5)
        mat = [[rng.randint(1, 4) for _ in range(n)] for _ in range(m)]
        X = max(min(r) for r in mat)
        Y = min(max(c) for c in zip(*mat))
        if X > Y:
            return False
    return True

assert answer_col_max([[1, 10, 4, 2], [9, 3, 8, 7], [15, 16, 17, 12]]) == [15, 16, 17, 12]
assert answer_lucky([[7, 8], [1, 2]]) == [7]
assert answer_lucky_o1([[1, 10, 4, 2], [9, 3, 8, 7], [15, 16, 17, 12]]) == [12]
assert answer_lucky_o1([[3, 6], [7, 1], [5, 2], [4, 8]]) == []
assert answer_saddle_points([[1, 1], [1, 1]]) == [(0, 0), (0, 1), (1, 0), (1, 1)]
assert answer_saddle_points([[2, 2], [1, 3]]) == [(0, 0)]
assert answer_check_inequality(300) is True

rng2 = random.Random(77)
for _ in range(200):
    mat = random_distinct_matrix(rng2, rng2.randint(1, 5), rng2.randint(1, 5))
    assert answer_lucky_o1(mat) == sorted(answer_lucky(mat)) == sorted(lucky_brute(mat))
print("Đáp án bài tập: PASS toàn bộ")

Đáp án bài tập: PASS toàn bộ


`float("inf")` / `float("-inf")` là **dương/âm vô cực** — giá trị khởi tạo tiện lợi cho bài tìm min/max: mọi số đều nhỏ hơn `inf` và lớn hơn `-inf`.

## 17. Tổng kết — template nhớ nhanh + checklist

```python
row_min = [min(r) for r in matrix]
col_max = [max(c) for c in zip(*matrix)]     # zip(*) = các cột
lucky = [v for i, r in enumerate(matrix) for j, v in enumerate(r)
         if v == row_min[i] == col_max[j]]

# Khi phần tử phân biệt: nhiều nhất 1 lucky
X, Y = max(row_min), min(col_max)
return [X] if X == Y else []
```

**Checklist trước khi submit**
- [ ] Đúng chiều: **min** hàng, **max** cột.
- [ ] Lấy cột bằng `zip(*matrix)` (có dấu `*`) hoặc chỉ số `matrix[i][j]`.
- [ ] Trả về **list** (kể cả rỗng `[]`).
- [ ] Mẹo `X == Y` chỉ dùng khi phần tử phân biệt.
- [ ] Đã test 1x1, một hàng, một cột, ma trận không có lucky.